In [ ]:
# Python code for grid-search for Fine-Tuning the CIFAR10 pretrained model

In [1]:
# Torch Library
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
from torch import optim
from torch import autograd
from torch.autograd import Variable
from torch.autograd import Function
from torch.utils.data import DataLoader
from torch.utils.data import random_split
from torch.utils.data.sampler import SubsetRandomSampler

# Torchvision Library
import torchvision
from torchvision import transforms
from torchvision import datasets
from torchvision.datasets import CIFAR10

# Common Library
import numpy as np
import random
import os

# Pretrained ResNet18 Library from huggingface 
# https://huggingface.co/docs/timm/en/index
import detectors
import timm

# Cuda Library
cudnn.benchmark = True
device = torch.device("cuda:0")

In [6]:
# Pre-Trained ResNet18 model

model = timm.create_model("resnet18_cifar10", pretrained=True)
model = model.to(device)

# ResNet18 config import

transform = timm.data.create_transform(
    **timm.data.resolve_data_config(model.pretrained_cfg)
)


In [7]:
## Data import

## Batch size
batch_size = 500
## Validation size
valid_size = 0.2

## Load the training and test datasets
train_data = datasets.CIFAR10('data', train=True,
                              download=True, transform=transform)
test_data = datasets.CIFAR10('data', train=False,
                             download=True, transform=transform)

# Indices for validation
num_train = len(train_data)
indices = list(range(num_train))
np.random.shuffle(indices)
split = int(np.floor(valid_size * num_train))
train_idx, valid_idx = indices[split:], indices[:split]

# Samplers for obtaining training and validation batches
train_sampler = SubsetRandomSampler(train_idx)
valid_sampler = SubsetRandomSampler(valid_idx)

# Data loaders
train_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size,
    sampler=train_sampler, num_workers=0)
valid_loader = torch.utils.data.DataLoader(train_data, batch_size=batch_size,
    sampler=valid_sampler, num_workers=0)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=batch_size,
    num_workers=0)

Files already downloaded and verified
Files already downloaded and verified


In [8]:
# Fine-Tuning optimizer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
# Target Image creation at evaluation

def evaluate_model(model, test_loader, stddev):
    # create test_examples with noise
    test_examples = []
    for images, labels in test_loader:
        nimages = images + torch.randn(images.size()) * stddev
        test_examples.append((nimages, labels))
        
    model.eval()  # Set the model to evaluation mode
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_examples:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = correct / total
    return accuracy

In [11]:
# Function to train the model with different noise levels 
# Mu is fixed at 1, delta is fixed at 0.00001

# Train each model until there is no update on validation loss

def train_with_noise(train_set, valid_set, epsilon_levels, alpha_levels, target_epsilon, target_alpha, waiting_epochs):
    
    # Fixed parameters
    delta = 0.00001
    mu = 1
    target_stddev =  np.sqrt(2*np.log(1.25/delta))*target_alpha*mu/target_epsilon

    ## target and validation set
    valid_examples = []
    for images, labels in valid_set:
        nimages = images + torch.randn(images.size()) * target_stddev
        valid_examples.append((nimages, labels))
        
    # Loop
    for epsilon in epsilon_levels:
        for alpha in alpha_levels:
            noise_level =  np.sqrt(2*np.log(1.25/delta))*alpha*mu/epsilon
            # create dirty training images
            training_examples = []
            # Loop over the test set and add Gaussian noise to each image
            for images, labels in train_set:
                # Add Gaussian noise to the image
                nimages = images + torch.randn(images.size()) * noise_level
                # Append the adversarial example and its label to the list
                training_examples.append((nimages, labels))
                # Append the original example and its label to the list
                training_examples.append((images, labels))
                
            # Initial validation loss
            valid_loss_min = np.Inf
            
            # load a clean Pre-Trained ResNet18 model
            
            model = timm.create_model("resnet18_cifar10", pretrained=True)
            model = model.to(device)
            
            n_epochs = waiting_epochs
            # Epochs
            for epoch in range(n_epochs+1):
                # Keep track of training and validation loss
                train_loss = 0.0
                valid_loss = 0.0
        
                # Train the model
                model.train()
                for images, labels in training_examples:
                    # Move data and target tensors to the default device
                    images, labels = images.to(device), labels.to(device)
                    
                    # Clear the gradients of all optimized variables
                    optimizer.zero_grad()
                    # Forward pass: compute predicted outputs by passing inputs to the model
                    output = model(images)
                    # Calculate the batch loss
                    loss = criterion(output, labels)
                    # Backward pass: compute gradient of the loss with respect to model parameters
                    loss.backward()
                    # Perform a single optimization step (parameter update)
                    optimizer.step()
                    # Update training loss
                    train_loss += loss.item()*images.size(0)
                    
                # Validate the model
                model.eval()
                for images, labels in valid_examples:
                    # Move data and target tensors to the default device
                    images, labels = images.to(device), labels.to(device)
                    # Forward pass: compute predicted outputs by passing inputs to the model
                    output = model(images)
                    # Calculate the batch loss
                    loss = criterion(output, labels)
                    # Update validation loss 
                    valid_loss += loss.item()*images.size(0)
                
                # Calculate average losses
                train_loss = train_loss/len(train_loader.dataset)
                valid_loss = valid_loss/len(valid_loader.dataset)
        
                    # Print training/validation statistics 
                print('Epoch: {} \tT_Loss: {:.6f} \tV_Loss: {:.6f}'.format(epoch, train_loss, valid_loss))
                
                # Save the model if validation loss has decreased
                if valid_loss <= valid_loss_min:
                    print('V_loss decreased ({:.6f} --> {:.6f}).  Saving model ...'.format(valid_loss_min, valid_loss))
                    torch.save(model.state_dict(), f'Tuning/model_cifar_ep_{epsilon}_al_{alpha}_target_ep_{target_epsilon}_target_al_{target_alpha}.pt')
                    valid_loss_min = valid_loss

In [13]:
## targeted IP constraints
delta = 0.00001
mu = 1

target_epsilon_levels = [0.5, 0.75, 1, 1.25, 1.5, 1.75, 2, 2.25, 2.5, 2.75, 3]
target_alpha_levels = [0.1]

# fixed search alpha_levels

# alpha_levels = [0.001, 0.01, 0.1]
alpha_levels = [0.1]

In [14]:
for target_epsilon in target_epsilon_levels:
    
    # generate training epsilon from target epsilon levels
    quarter_epsilon = target_epsilon * 0.25
    half_epsilon = target_epsilon * 0.5
    high_epsilon = target_epsilon * 1.15
    low_epsilon = target_epsilon * 0.85
    twice_epsilon = target_epsilon * 2
    epsilon_levels = [quarter_epsilon, half_epsilon, low_epsilon, target_epsilon, high_epsilon, twice_epsilon]
    #epsilon_levels = [half_epsilon]

    for target_alpha in target_alpha_levels:
        train_with_noise(train_loader, valid_loader, epsilon_levels, alpha_levels, target_epsilon, target_alpha, 10)


Epoch: 0 	T_Loss: 3.938967 	V_Loss: 0.703615
V_loss decreased (inf --> 0.703615).  Saving model ...
Epoch: 1 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 2 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 3 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 4 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 5 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 6 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 7 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 8 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 9 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 10 	T_Loss: 3.938967 	V_Loss: 0.703616
Epoch: 0 	T_Loss: 3.388894 	V_Loss: 0.743995
V_loss decreased (inf --> 0.743995).  Saving model ...
Epoch: 1 	T_Loss: 3.388894 	V_Loss: 0.743994
V_loss decreased (0.743995 --> 0.743994).  Saving model ...
Epoch: 2 	T_Loss: 3.388894 	V_Loss: 0.743994
V_loss decreased (0.743994 --> 0.743994).  Saving model ...
Epoch: 3 	T_Loss: 3.388894 	V_Loss: 0.743994
V_loss decreased (0.743994 --> 0.743994).  Saving model ...
Epoch: 4 	T_Loss: 3.388894 	V_Loss

In [17]:
for target_epsilon in target_epsilon_levels:
    
    # generate training epsilon from target epsilon levels
    quarter_epsilon = target_epsilon * 0.25
    half_epsilon = target_epsilon * 0.5
    high_epsilon = target_epsilon * 1.15
    low_epsilon = target_epsilon * 0.85
    twice_epsilon = target_epsilon * 2
    epsilon_levels = [quarter_epsilon, half_epsilon, low_epsilon, target_epsilon, high_epsilon, twice_epsilon]
    # epsilon_levels = [half_epsilon]
    
    for target_alpha in target_alpha_levels:
    
        for epsilon in epsilon_levels:
        
            for alpha in alpha_levels: 
                
                print(f'tuning_noise_ep_{epsilon}_al_{alpha}_target_ep_{target_epsilon}_target_al_{target_alpha}')
            
                model = timm.create_model("resnet18_cifar10", pretrained=True)
                model = model.to(device)
                model.load_state_dict(torch.load(f'Tuning/model_cifar_ep_{epsilon}_al_{alpha}_target_ep_{target_epsilon}_target_al_{target_alpha}.pt'))
                
                target_stddev =  np.sqrt(2*np.log(1.25/delta))*target_alpha*mu/target_epsilon

                targeted_accuracy = evaluate_model(model, test_loader, target_stddev )
                print(f'tuned_model_on_targeted_accuracy_is_{targeted_accuracy}')
                clean_accuracy= evaluate_model(model, test_loader, 0)
                print(f'clean_accuracy_is_{clean_accuracy}')
                print('')


tuning_noise_ep_0.125_al_0.1_target_ep_0.5_target_al_0.1
tuned_model_on_targeted_accuracy_is_0.3593
clean_accuracy_is_0.212

tuning_noise_ep_0.25_al_0.1_target_ep_0.5_target_al_0.1
tuned_model_on_targeted_accuracy_is_0.3409
clean_accuracy_is_0.4238

tuning_noise_ep_0.425_al_0.1_target_ep_0.5_target_al_0.1
tuned_model_on_targeted_accuracy_is_0.2549
clean_accuracy_is_0.6499

tuning_noise_ep_0.5_al_0.1_target_ep_0.5_target_al_0.1
tuned_model_on_targeted_accuracy_is_0.2355
clean_accuracy_is_0.6929

tuning_noise_ep_0.575_al_0.1_target_ep_0.5_target_al_0.1
tuned_model_on_targeted_accuracy_is_0.2056
clean_accuracy_is_0.7366

tuning_noise_ep_1.0_al_0.1_target_ep_0.5_target_al_0.1
tuned_model_on_targeted_accuracy_is_0.1596
clean_accuracy_is_0.8451

tuning_noise_ep_0.1875_al_0.1_target_ep_0.75_target_al_0.1
tuned_model_on_targeted_accuracy_is_0.5598
clean_accuracy_is_0.3188

tuning_noise_ep_0.375_al_0.1_target_ep_0.75_target_al_0.1
tuned_model_on_targeted_accuracy_is_0.5202
clean_accuracy_is_0.5

In [18]:
torch.cuda.empty_cache()